In [16]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError("GOOGLE_API_KEY bulunamadı!")

# Geçerli model adı kullanın
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

messages = [
    HumanMessage(content="Bana Yapay Zeka tarihi hakkında şaşırtıcı, tek cümlelik bir bilgi ver.")
]

try:
    print("Gemini Düşünüyor...")
    result = model.invoke(messages)
    print("Cevap:", result.content)
except Exception as e:
    print("Hata:", e)

Gemini Düşünüyor...
Cevap: Modern bilgisayarların icadından çok önce, 13. yüzyılda yaşamış İspanyol rahip Ramon Llull, mantıksal işlemleri mekanik olarak taklit edebilen bir "düşünme makinesi" tasarlamıştı.


In [18]:
# Önce kütüphaneyi kuralım:
# pip install guidance

import guidance
from guidance import models, select

# 1. Mock (Taklit) Model Oluşturma
# Gerçek bir LLM yerine, sanki bir LLM varmış gibi davranan simülatör.
# Bu model, bizim verdiğimiz metni 'biliyormuş' gibi davranacak.
lm = models.Mock()

# 2. Guidance Programı Yazma
# Modelden bir 'Karakter' seçmesini istiyoruz.
# 'select' fonksiyonu, modelin çıktısını SADECE verilen listeyle sınırlar.

with guidance.user():
    lm += "Yüzüklerin Efendisi serisindeki en sevdiğin karakter kim?"

with guidance.assistant():
    lm += "Benim favorim " + select(["Gandalf", "Aragorn", "Frodo"], name="karakter") + " çünkü o çok cesur."

print("-" * 30)
print("Guidance Çıktısı:")
print(f"Seçilen Karakter: {lm['karakter']}")
print("-" * 30)
print("Tam Metin:\n" + str(lm))

------------------------------
Guidance Çıktısı:
Seçilen Karakter: Gandalf
------------------------------
Tam Metin:
<|im_start|>user
Yüzüklerin Efendisi serisindeki en sevdiğin karakter kim?<|im_end|>
<|im_start|>assistant
Benim favorim Gandalf çünkü o çok cesur.


In [24]:
import dspy

class BenimMockModel(dspy.LM):
    def __init__(self):
        super().__init__("mock-model")
        self.history = []

    # DSPy versiyonuna göre bazen 'prompt' bazen 'messages' gelir.
    def __call__(self, prompt=None, messages=None, **kwargs):
        # Gelen girdi hangisiyse onu yakalayalım
        input_data = prompt if prompt else str(messages)
        
        # Geçmişe kaydedelim (Böylece DSPy'ın ürettiği promptu görebiliriz)
        self.history.append({'prompt': input_data, 'response': 'Paris'})
        
        # DSPy bir liste içinde string cevap bekler
        return ["Paris"]

# -----------------------------------------------------------

# 1. Görevi Tanımla (İmza)
class BasitSoruCevap(dspy.Signature):
    """Kısa ve öz cevaplar veren bir asistan."""
    soru = dspy.InputField(desc="Kullanıcının merak ettiği konu")
    cevap = dspy.OutputField(desc="Bilgilendirici yanıt")

# 2. Modeli Ayarla
mock_model = BenimMockModel()
dspy.settings.configure(lm=mock_model)

# 3. Modülü Oluştur (ChainOfThought)
# Biz sadece 'soru' ve 'cevap' istedik. 
# Bakalım DSPy araya 'Reasoning' (Akıl Yürütme) ekleyecek mi?
program = dspy.ChainOfThought(BasitSoruCevap)

# 4. Çalıştır
# Hata almamak için try-except bloğu ile (mock model olduğu için bazen parser hatası verebilir, ama promptu üretir)
try:
    sonuc = program(soru="Fransa'nın başkenti neresidir?")
except:
    pass # Cevap 'Paris' olduğu için parser bazen formatı beğenmeyebilir, önemli olan promptu görmek.

# 5. DSPy'ın Arka Planda Ürettiği Promptu İncele
print("-" * 30)
print("DSPy Tarafından Otomatik Üretilen Prompt:")
print("-" * 30)

# Yakalanan son veriyi yazdır
if mock_model.history:
    print(mock_model.history[-1]['prompt'])
else:
    print("Geçmiş boş.")
    
print("-" * 30)

------------------------------
DSPy Tarafından Otomatik Üretilen Prompt:
------------------------------
[{'role': 'system', 'content': 'Your input fields are:\n1. `soru` (str): Kullanıcının merak ettiği konu\nYour output fields are:\n1. `reasoning` (str): \n2. `cevap` (str): Bilgilendirici yanıt\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\nInputs will have the following structure:\n\n[[ ## soru ## ]]\n{soru}\n\nOutputs will be a JSON object with the following fields.\n\n{\n  "reasoning": "{reasoning}",\n  "cevap": "{cevap}"\n}\nIn adhering to this structure, your objective is: \n        Kısa ve öz cevaplar veren bir asistan.'}, {'role': 'user', 'content': "[[ ## soru ## ]]\nFransa'nın başkenti neresidir?\n\nRespond with a JSON object in the following order of fields: `reasoning`, then `cevap`."}]
------------------------------
